In [2]:
# ================================
# STEP 1: Load Dataset
# ================================
import pandas as pd

df = pd.read_csv(r"C:\Users\shett\Downloads\fortune cleaned.csv")

# Combine important fields into text
df["text"] = df.apply(lambda row: 
    f"{row['Company']} is in {row['Sector']} sector, led by {row['CEO']} located in {row['HeadquartersCity']}, {row['HeadquartersState']}", axis=1)

texts = df["text"].tolist()

print(texts[:2])


# ================================
# STEP 2: Embeddings
# ================================
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(texts)

print(len(embeddings), len(embeddings[0]))


# ================================
# STEP 3: FAISS (Local Search)
# ================================
import faiss
import numpy as np

dimension = len(embeddings[0])

faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(embeddings))

print("Total vectors:", faiss_index.ntotal)


def semantic_search(query, k=3):
    query_embedding = model.encode([query])
    D, I = faiss_index.search(query_embedding, k)
    return [texts[i] for i in I[0]]


print(semantic_search("technology companies"))


# ================================
# STEP 4: Pinecone Setup
# ================================
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_4d4HZw_6oSoCyvY9d26nptmnujm5wPVfU7BckK84FHprjVmXa6KoodFj8FySJiNRF14gf4")   # 🔴 Replace with your key

index_name = "company-search"

# Create index (only once)
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("✅ Connected to Pinecone")


# ================================
# STEP 5: Store Data in Pinecone
# ================================
vectors = []

for i, emb in enumerate(embeddings):
    vectors.append((
        str(i),
        emb.tolist(),
        {"text": texts[i]}
    ))

index.upsert(vectors)

print("✅ Data stored in Pinecone")


# ================================
# STEP 6: Pinecone Search
# ================================
def pinecone_search(query, k=3):
    query_vector = model.encode([query]).tolist()

    results = index.query(
        vector=query_vector[0],
        top_k=k,
        include_metadata=True
    )

    return [match['metadata']['text'] for match in results['matches']]


# ================================
# STEP 7: Load LLM (GPT-2)
# ================================
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")


# ================================
# STEP 8: RAG FUNCTION (FIXED)
# ================================
def rag_llm(query):

    # Step 1: Retrieve
    docs = pinecone_search(query)

    # Improve accuracy
    if "technology" in query.lower():
        docs = [d for d in docs if "Technology sector" in d]

    # Step 2: Context
    context = "\n".join(docs)

    # ✅ FIXED PROMPT
    prompt = f"""
You are an AI assistant.

From the context below, answer the question accurately.

Context:
{context}

Question: {query}

Give a short and clear answer. Only include company names if asked.

Answer:
"""

    # Generate
    result = generator(prompt, max_new_tokens=80, num_return_sequences=1)

    # Clean output
    output = result[0]['generated_text']

    if "Answer:" in output:
        answer = output.split("Answer:")[-1].strip()
    else:
        answer = output.strip()

    answer = answer.split("\n")[0]

    return answer


# ================================
# STEP 9: TEST
# ================================
query = "Which companies are in technology sector?"

print("\n🔍 Query:", query)
print("✅ Answer:", rag_llm(query))

['Walmart is in Retailing sector, led by C. Douglas Mcmillon located in Bentonville, Arkansas', 'Amazon is in Retailing sector, led by Andrew R. Jassy located in Seattle, Washington']


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1000 384
Total vectors: 1000
['Cognizant Technology Solutions is in Technology sector, led by Ravi Kumar S located in Teaneck, New Jersey', 'Ss&C Technologies Holdings is in Technology sector, led by William C. Stone located in Windsor, Connecticut', 'Enterprise Products Partners is in Energy sector, led by A. James Teague/W. Randall Fowler located in Houston, Texas']
✅ Connected to Pinecone
✅ Data stored in Pinecone


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


🔍 Query: Which companies are in technology sector?


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Answer: Nokia, Microsoft, Cisco, Intel, Lenovo, Samsung, HP, HPE, Motorola, Intel, Huawei, Huawei, Microsoft, Samsung, Lenovo, Microsoft, Motorola, Intel, Nokia, Motorola, Lenovo, Lenovo, Lenovo, Lenovo, Lenovo, Lenovo, Qualcomm, Motorola, Intel, Lenovo, Lenovo, Lenovo, Lenovo, Lenovo, Qualcomm, Motorola, Intel, Lenovo, Lenovo
